# 07 - Passing Network Generator & Tactical Heatmap Visualizer

## Executive Summary & Objectives
This notebook implements structural network topology and spatial density heatmaps:
1. **Passing Network Generator**:
   - Derives player spatial centroids ($x, y$) during possession phases.
   - Computes directed pair-wise pass volumes and connection thickness.
   - Calculates network centrality metrics (degree centrality, node connectivity).
   - Visualizes tactical formation structure and passing channels on a pitch.
2. **Spatial Heatmap Generator**:
   - 2D Gaussian Kernel Density Estimation (KDE) and binned 2D pitch histograms.
   - Action density heatmaps, defensive regains, and player touch footprints.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

DATA_DIR = '../data' if os.path.exists('../data') else 'data'
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

print("Initializing Passing Network & Heatmap Engine...")

## 1. Passing Network Generator Implementation
We extract consecutive completed passes between teammates in the same match to build nodes (player centroids) and edges (pass volume).

In [ ]:
actions_path = os.path.join(DATA_DIR, 'actions.csv')
df_act = pd.read_csv(actions_path, nrows=40000)

def generate_passing_network(df, match_id=None, team_id=None):
    df_m = df.copy()
    if match_id is not None:
        df_m = df_m[df_m['game_id'] == match_id]
    if team_id is not None:
        df_m = df_m[df_m['team_id'] == team_id]
    else:
        team_id = df_m['team_id'].iloc[0]
        df_m = df_m[df_m['team_id'] == team_id]
        
    is_pass = (df_m['type_name'] == 'pass') | (df_m['type_id'] == 0)
    is_succ = is_pass & ((df_m['result_name'] == 'success') | (df_m['result_id'] == 1))
    df_passes = df_m[is_succ].copy().reset_index(drop=True)
    
    # Compute Player Node Centroids
    nodes = df_passes.groupby('player_id').agg(
        pos_x=('start_x', 'mean'),
        pos_y=('start_y', 'mean'),
        n_passes=('action_id', 'count')
    ).reset_index()
    
    # Compute Edge Connections (Consecutive passes between player A and player B)
    df_passes['next_player_id'] = df_passes['player_id'].shift(-1)
    df_passes['time_diff'] = df_passes['time_seconds'].shift(-1) - df_passes['time_seconds']
    
    # Edges within same sequence (< 5s)
    valid_edges = df_passes[(df_passes['time_diff'] < 5.0) & (df_passes['player_id'] != df_passes['next_player_id'])]
    edges = valid_edges.groupby(['player_id', 'next_player_id']).size().reset_index(name='pass_count')
    
    return nodes, edges

sample_game = df_act['game_id'].iloc[0]
sample_team = df_act['team_id'].iloc[0]
nodes, edges = generate_passing_network(df_act, match_id=sample_game, team_id=sample_team)

print(f"Passing Network for Team {sample_team} in Match {sample_game}:")
print(f"- Active Nodes (Players): {len(nodes)}")
print(f"- Pairwise Passing Edges: {len(edges)}")
display(nodes.head(5))
display(edges.sort_values('pass_count', ascending=False).head(5))

## 2. Visualizing Tactical Passing Network on Pitch
Plotting player nodes sized by pass volume and lines with width proportional to pairwise pass counts.

In [ ]:
plt.figure(figsize=(10, 6.5))

# Draw pitch outline
plt.plot([0, 105, 105, 0, 0], [0, 0, 68, 68, 0], color='gray')
plt.axvline(52.5, color='gray', linestyle='--')

# Draw edges
node_dict = {row['player_id']: (row['pos_x'], row['pos_y']) for _, row in nodes.iterrows()}

for _, edge in edges.iterrows():
    p1, p2, count = edge['player_id'], edge['next_player_id'], edge['pass_count']
    if p1 in node_dict and p2 in node_dict and count >= 2:
        x1, y1 = node_dict[p1]
        x2, y2 = node_dict[p2]
        plt.plot([x1, x2], [y1, y2], color='#1f77b4', alpha=0.6, linewidth=count * 0.8)

# Draw nodes
plt.scatter(nodes['pos_x'], nodes['pos_y'], s=nodes['n_passes']*15 + 100, color='#d62728', edgecolors='black', zorder=5)
for _, row in nodes.iterrows():
    plt.text(row['pos_x'] + 1, row['pos_y'] + 1, f"P{int(row['player_id'])}", fontsize=9, fontweight='bold')

plt.title(f'Tactical Passing Network - Team {sample_team}')
plt.xlabel('X Coordinate (Attacking Left -> Right)')
plt.ylabel('Y Coordinate')
plt.xlim(-2, 107)
plt.ylim(-2, 70)
plt.tight_layout()
plt.show()

## 3. Spatial Heatmap Generator (2D KDE & Action Density)
Generate continuous Kernel Density Estimation heatmaps for team/player spatial territorial footprints.

In [ ]:
def plot_spatial_heatmap(df_actions, player_id=None, title='Spatial Density Heatmap'):
    df_sub = df_actions if player_id is None else df_actions[df_actions['player_id'] == player_id]
    
    plt.figure(figsize=(10, 6.5))
    sns.kdeplot(
        x=df_sub['start_x'],
        y=df_sub['start_y'],
        fill=True,
        cmap='magma',
        thresh=0.05,
        levels=15,
        alpha=0.8
    )
    plt.plot([0, 105, 105, 0, 0], [0, 0, 68, 68, 0], color='white', linewidth=1.5)
    plt.axvline(52.5, color='white', linestyle='--', alpha=0.7)
    plt.title(title, fontsize=14)
    plt.xlim(0, 105)
    plt.ylim(0, 68)
    plt.xlabel('X (m)')
    plt.ylabel('Y (m)')
    plt.tight_layout()
    plt.show()

# Sample Heatmap for top active player
top_player = df_act['player_id'].value_counts().index[0]
plot_spatial_heatmap(df_act, player_id=top_player, title=f'Spatial Touch & Action Heatmap for Player #{top_player}')